# Coding Attention Mechanism

## A simple self-attention mechanism without trainable weights

In [1]:
import torch

In [2]:
inputs = torch.tensor([
        [0.33, 0.17, 0.16], # My      (x^1)
        [0.91, 1.50, 1.30], # name    (x^2)
        [1.27, 0.20, 0.16], # is      (x^3)
        [0.45, 0.96, 1.11], # khalid  (x^4) 
        [1.19, 0.32, 0.63], # khan    (x^5)
        [2.80, 0.78, 1.46]  # kakar   (x^6)
    ])


In [3]:
input_query = inputs[1]
input_query

tensor([0.9100, 1.5000, 1.3000])

In [4]:
input_0 = inputs[0]
input_0

tensor([0.3300, 0.1700, 0.1600])

In [5]:
# 0.3300 * 0.9100 + 0.1700 * 1.5000 + 0.1600 * 1.3000
torch.dot(input_query, input_0)

tensor(0.7633)

In [6]:
res = 0.
i = 3

for idx, ele in enumerate(inputs[i]):
    res += inputs[i][idx] * input_query[idx]
res
print('Pythonic way --> ', res)
# or

res = torch.dot(inputs[i], input_query)

print('Torch way --> ', res)


Pythonic way -->  tensor(3.2925)
Torch way -->  tensor(3.2925)


In [7]:
# All the attention score with respect to input_query


atten_score = torch.empty(inputs.shape[0])

for i, i_x in enumerate(inputs):
    atten_score[i] = torch.dot(i_x, input_query)


atten_score = atten_score / atten_score.sum() # Normalization


In [8]:
def softmax_navie(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
softmax_navie(atten_score).sum()


tensor(1.0000)

In [9]:
# atten_score = torch.nn.Softmax(atten_score)
# atten_score

In [10]:
# context vector 

context_vec = torch.zeros(input_query.shape)

for i, i_x in enumerate(inputs):
    # print(f"{i_x} ---> {atten_score[i]}")
    context_vec += i_x * atten_score[i]

context_vec

tensor([1.4468, 0.8611, 1.0788])

In [11]:
# calculate the it for all

# Method 1
atten_scores = torch.empty(6, 6)
for i, i_x in enumerate(inputs):
    for j, j_x in enumerate(inputs):
        atten_scores[i, j] = torch.dot(i_x, j_x)
atten_scores

tensor([[ 0.1634,  0.7633,  0.4787,  0.4893,  0.5479,  1.2902],
        [ 0.7633,  4.7681,  1.6637,  3.2925,  2.3819,  5.6160],
        [ 0.4787,  1.6637,  1.6785,  0.9411,  1.6761,  3.9456],
        [ 0.4893,  3.2925,  0.9411,  2.3562,  1.5420,  3.6294],
        [ 0.5479,  2.3819,  1.6761,  1.5420,  1.9154,  4.5014],
        [ 1.2902,  5.6160,  3.9456,  3.6294,  4.5014, 10.5800]])

In [12]:
# Method 2
atten_scores = inputs @ inputs.T
atten_scores

tensor([[ 0.1634,  0.7633,  0.4787,  0.4893,  0.5479,  1.2902],
        [ 0.7633,  4.7681,  1.6637,  3.2925,  2.3819,  5.6160],
        [ 0.4787,  1.6637,  1.6785,  0.9411,  1.6761,  3.9456],
        [ 0.4893,  3.2925,  0.9411,  2.3562,  1.5420,  3.6294],
        [ 0.5479,  2.3819,  1.6761,  1.5420,  1.9154,  4.5014],
        [ 1.2902,  5.6160,  3.9456,  3.6294,  4.5014, 10.5800]])

In [13]:
# Normalization
atten_weigths = torch.softmax(atten_scores, dim=1)
atten_weigths

tensor([[9.8692e-02, 1.7981e-01, 1.3527e-01, 1.3672e-01, 1.4497e-01, 3.0454e-01],
        [4.9020e-03, 2.6893e-01, 1.2062e-02, 6.1489e-02, 2.4736e-02, 6.2788e-01],
        [2.2458e-02, 7.3454e-02, 7.4550e-02, 3.5661e-02, 7.4371e-02, 7.1951e-01],
        [1.9414e-02, 3.2029e-01, 3.0503e-02, 1.2558e-01, 5.5629e-02, 4.4859e-01],
        [1.4473e-02, 9.0585e-02, 4.4723e-02, 3.9110e-02, 5.6814e-02, 7.5430e-01],
        [9.1299e-05, 6.9046e-03, 1.2992e-03, 9.4704e-04, 2.2650e-03, 9.8849e-01]])

In [14]:
# context vector for all
all_ctx_vec = atten_weigths @ inputs
all_ctx_vec


tensor([[1.4547, 0.7287, 0.9589],
        [2.0768, 0.9633, 1.3529],
        [2.2881, 0.7482, 1.2479],
        [1.7154, 0.9781, 1.2537],
        [2.3412, 0.7914, 1.3077],
        [2.7789, 0.7833, 1.4549]])

In [15]:
# summary and Clean 
atten_scores = inputs @ inputs.T # calculate the attention scores for all vectors 
atten_weigths = torch.softmax(atten_scores, dim=1) # normalize the attention scores
context_vector = atten_weigths @ inputs # compute the context vector for all

# Next Section: Self attention with trainable weights

## Compute the attention weights step by step

In [16]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [17]:
torch.manual_seed(123)
# Parameter class has weights which can be later trainable weights 
W_query = torch.nn.Parameter(torch.rand(d_in, d_out)) 
W_key = torch.nn.Parameter(torch.rand(d_in, d_out)) 
W_value = torch.nn.Parameter(torch.rand(d_in, d_out)) 

In [18]:
query_2 = x_2 @ W_query 
query_2

tensor([0.7431, 2.6294], grad_fn=<SqueezeBackward4>)

In [19]:
# Note: Keys and Values are unique with respect to each, but query will same as inputs vector changes 
keys = inputs @ W_key
values = inputs @ W_value
keys, values

(tensor([[0.1268, 0.2673],
         [0.8102, 2.0762],
         [0.2607, 0.3854],
         [0.5881, 1.5062],
         [0.4200, 0.7873],
         [0.9863, 1.8567]], grad_fn=<MmBackward0>),
 tensor([[0.0977, 0.2656],
         [0.6976, 1.8572],
         [0.1783, 0.4625],
         [0.4694, 1.3926],
         [0.2660, 0.8838],
         [0.6317, 2.0719]], grad_fn=<MmBackward0>))

In [20]:
keys_2 = keys[1]
atten_score_22 = torch.dot(keys_2, query_2)
atten_score_22

tensor(6.0611, grad_fn=<DotBackward0>)

In [21]:
#  for all
atten_score_2 =   query_2 @ keys.T
atten_score_2

tensor([0.7969, 6.0611, 1.2070, 4.3974, 2.3822, 5.6150],
       grad_fn=<SqueezeBackward4>)

In [27]:
d_k = keys.shape[1]

atten_weigths_2 =  torch.softmax(atten_score_2 / d_k ** 0.5, dim=-1)
atten_weigths_2

tensor([0.0111, 0.4611, 0.0149, 0.1422, 0.0342, 0.3364],
       grad_fn=<SoftmaxBackward0>)

In [28]:
context_vec_2 = atten_weigths_2 @ values
context_vec_2

tensor([0.6138, 1.7915], grad_fn=<SqueezeBackward4>)

## A compact class of Self Attention

In [ ]:
import torch.nn as nn

class SelfAttentionV1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()

        self.d_in = d_in
        self.d_out = d_out

        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out)) 
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out)) 
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out)) 

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        d_k = keys.shape[1]

        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores / d_k ** 0.5, dim=-1)
        context_vec = atten_weights @ values

        return context_vec



torch.manual_seed(123)

d_in = inputs.shape[1]
d_out = 2   

sa_v1 = SelfAttentionV1(d_in, d_out)
sa_v1(inputs)

tensor([[0.4442, 1.3149],
        [0.6138, 1.7915],
        [0.5063, 1.4964],
        [0.5746, 1.6861],
        [0.5444, 1.6037],
        [0.6340, 1.8444]], grad_fn=<MmBackward0>)

In [32]:
import torch.nn as nn

# Adding the Linear layer which has weights and bias matrix
class SelfAttentionV2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.d_in = d_in
        self.d_out = d_out

        self.W_query = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in, d_out , bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x )

        d_k = keys.shape[1]

        atten_scores = queries @ keys.T
        atten_weights = torch.softmax(atten_scores / d_k ** 0.5, dim=-1)
        context_vec = atten_weights @ values

        return context_vec



torch.manual_seed(123)

d_in = inputs.shape[1]
d_out = 2   

sa_v1 = SelfAttentionV2(d_in, d_out)
sa_v1(inputs)

tensor([[-0.9982, -0.1178],
        [-1.0931, -0.1235],
        [-1.0660, -0.1205],
        [-1.0662, -0.1215],
        [-1.1052, -0.1217],
        [-1.3089, -0.1263]], grad_fn=<MmBackward0>)